In [3]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
from functions import clean_data, split_data

In [7]:
X, Y = clean_data("claims_train.csv")
X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)

In [ ]:
def build_model():
    model = keras.Sequential([
        keras.layers.Dense(28, activation = "relu", input_shape = (41,), kernel_initializer = "he_normal"),
        keras.layers.Dense(28, activation = "relu",  kernel_initializer = "he_normal"),
        keras.layers.Dense(1, activation = "softplus",  kernel_initializer = "he_normal")
    ])

    model.compile(
        optimizer =keras.optimizers.SGD(learning_rate = 0.001),
        loss = keras.losses.Huber(),
        metrics=["mse"]
    )

    return model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',     # what to monitor
    patience=5,             # how many epochs with no improvement
    restore_best_weights=True
)

In [ ]:
batch_sizes = [128, 256, 512]
epochs = [50, 70, 100] 

results = []

for batch in batch_sizes:
    for epoch in epochs:
        model = build_model()
        
        history = model.fit(
            X_train, y_train,
            batch_size = batch,
            epochs = epoch,
            validation_data = (X_val, y_val),
            callbacks=[early_stop], 
            verbose = 0
        )
        
        validation_mse = history.history["val_mse"][-1]
        
        results.append((batch, epoch, validation_mse))
        print(f"batch={batch}, epochs={epoch} => val MSE={validation_mse:.4f}")

In [ ]:
results = sorted(results, key=lambda x: x[2])
best_batch, best_epoch, best_val_mse = results[0]

print("Best parameters:")
print("Batch size:", best_batch)
print("Epochs:", best_epoch)
print("Validation MSE:", best_val_mse)

In [ ]:
chosen_model = build_model()
final = chosen_model.fit(
    X_train, y_train,
    batch_size = best_batch,
    epochs = best_epoch,
    validation_data = (X_val, y_val),
    callbacks=[early_stop],
    verbose = 1
    )

In [ ]:
# test_loss, test_mse = chosen_model.evaluate(X_test, y_test)
# print("Test MSE:", test_mse)